# Frontier LLM Research Engineering — Phase 0 scaled pilots

Run on Colab with a GPU runtime (**A100** preferred; **L4** for the FP8 pilot; T4 for the free-GPU
checks). Run the cells top to bottom. Every pilot writes to Google Drive and resumes if the session
disconnects: just run the same cell again.

Order matters when compute units run out: **P1 (noise floor) first**, then the others.

In [ ]:
# 1. Mount Drive (outputs survive disconnects) and choose where pilots write.
from google.colab import drive
drive.mount('/content/drive')
PILOT_DIR = '/content/drive/MyDrive/frontier-llm-pilots'
import os; os.makedirs(PILOT_DIR, exist_ok=True); os.environ['PILOT_DIR'] = PILOT_DIR
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 2. Get the course code. Option A: the GitHub repo (set REPO_URL; a private repo needs a token in the URL).
#    Option B: upload a zip of the repo to Drive and set ZIP_PATH.
REPO_URL = 'https://github.com/tal-giladi/frontier-llm-research-course.git'
ZIP_PATH = ''   # e.g. '/content/drive/MyDrive/frontier-llm-research-course.zip'
import os
if ZIP_PATH:
    !rm -rf /content/course && mkdir -p /content/course && unzip -q "$ZIP_PATH" -d /content/course
    ROOT = '/content/course/' + os.listdir('/content/course')[0] if len(os.listdir('/content/course')) == 1 else '/content/course'
else:
    !rm -rf /content/course && git clone -q "$REPO_URL" /content/course
    ROOT = '/content/course'
%cd $ROOT
!git log --oneline -1 2>/dev/null || true

In [ ]:
# 3. Install the pinned stack (references/versions.md). PyTorch 2.14.1 CUDA wheel, then the lab package.
!pip -q install torch==2.14.1 --index-url https://download.pytorch.org/whl/cu128
!pip -q install numpy==2.5.3 scipy transformers==5.18.0 safetensors tokenizers==0.23.2 datasets==5.0.1 matplotlib==3.11.2 pytest==9.1.1 pyyaml==6.0.3 torchao==0.18.0
!pip -q install -e labs/common
import torch; print(torch.__version__, torch.cuda.get_device_name(0))
!python -m pytest labs/common/tests/test_model.py -x -q 2>&1 | tail -3   # quick check; the full suite runs on CPU in the build

In [ ]:
# 4. Data-v0 at pilot size (vocab 32768). ~800k documents is ~0.8B train tokens: enough for the P1 ladder.
#    Writes to Drive; skipped if already there. Expect roughly 30-60 minutes and ~2 GB.
DATA = f'{PILOT_DIR}/data-v0-32k'
import os
if not os.path.exists(f'{DATA}/meta.json'):
    !python -m frontierlab.data.prepare --docs 800000 --vocab 32768 --out "$DATA"
# Lab data and every runs/ folder live on Drive, so a disconnect or a new session loses nothing.
!mkdir -p "$PILOT_DIR/labdata" "$PILOT_DIR/runs" && rm -rf labs/common/data runs
!ln -sfn "$PILOT_DIR/labdata" labs/common/data && ln -sfn "$PILOT_DIR/runs" runs
!rm -rf labs/common/data/v0 && ln -sfn "$DATA" labs/common/data/v0
!cat "$DATA/meta.json" | head -40

## P1b — Module 1 lesson labs on GPU (about 30 minutes)

01.3 comparison axes, 01.4 seeds, 01.5 batch invariance and bitwise reruns. Record what each script
prints; outputs go to Drive.

In [ ]:
os.environ['LAB_TARGET'] = 'solution'
O1 = f'{PILOT_DIR}/p1b'; os.makedirs(O1, exist_ok=True)
!python curriculum/pilots/once.py P1B 0 'python labs/module-01/lesson-03/compare_axes.py --device cuda --batch 64 --seq 512 --steps 2000 2>&1 | tee "$O1/01-3-axes.txt"'
!python curriculum/pilots/once.py P1B 1 'python labs/module-01/lesson-04/run_seeds.py --preset pilot-10m --device cuda --batch 64 --seq 512 --steps 2000 --out "$O1/l14-gpu" 2>&1 | tail -20'
!python curriculum/pilots/once.py P1B 2 'python labs/module-01/lesson-04/analyze.py --out "$O1/l14-gpu" 2>&1 | tee "$O1/01-4-analyze.txt"'
!python curriculum/pilots/once.py P1B 3 'python labs/module-01/lesson-05/batch_invariance.py --device cuda 2>&1 | tee "$O1/01-5-batch-invariance.txt"'

In [ ]:
# Last cell: marks P1B complete when no command of it has failed; the status notebook reads this.
import glob, os
failed = glob.glob(f'{PILOT_DIR}/.failed/P1B-*')
if failed:
    print('P1B NOT complete; failed commands (rerun this notebook to retry):', failed)
else:
    open(f'{PILOT_DIR}/.done/P1B-COMPLETE', 'w').write('ok')
    print('P1B complete')